# Creating OGC Dimensions

This notebook demonstrates how to create and use **OGC Dimension providers** — the building blocks for paginated, machine-discoverable datacube dimensions in STAC and OGC API Records.

Three provider types are covered:

| Provider | Use case | Examples |
|-----------|----------|----------|
| `DailyPeriodProvider` | Calendar-based temporal periods | Dekadal (10-day), Pentadal (5-day) |
| `LeveledTreeProvider` | Hierarchical nominal dimensions | Admin boundaries, indicator catalogs, species taxonomy |
| `IntegerRangeProvider` | Numeric sequences | Elevation bands, percentile classes |

Spec: [github.com/ccancellieri/ogc-dimensions](https://github.com/ccancellieri/ogc-dimensions)

## 1. Dekadal Generator (10-day periods)

A **dekad** divides each month into three periods:
- D1: days 1-10
- D2: days 11-20
- D3: days 21-end of month (absorbs remainder: 8, 9, 10, or 11 days)

This gives 36 periods per year. Used by FAO ASIS, FEWS NET, TUW-GEO.

In [ ]:
from ogc_dimensions.providers import DailyPeriodProvider

dekadal = DailyPeriodProvider(period_days=10, scheme="monthly")

print(f"Provider type:  {dekadal.provider_type}")
print(f"Config:         {dekadal.config_as_dict()}")
print(f"Invertible:     {dekadal.invertible}")
print(f"Capabilities:   {dekadal.capabilities}")
print(f"Search protos:  {dekadal.search_protocols}")

In [ ]:
# Generate members for Q1 2025
result = dekadal.generate("2025-01-01", "2025-03-31", limit=12)

print(f"Total matched: {result.number_matched}")
print(f"Returned:      {len(result.members)}")
print()
for m in result.members:
    print(f"  {m.code}  {m.start} -> {m.end}")

In [ ]:
# Descending order — most recent first
desc = dekadal.generate("2025-01-01", "2025-03-31", limit=3, sort_dir="desc")
for m in desc.members:
    print(f"  {m.code}  {m.start} -> {m.end}")

### Inverse Lookup

Given an arbitrary date, find which dekad it belongs to:

In [ ]:
inv = dekadal.inverse("2025-01-15")
print(f"Date 2025-01-15 -> member {inv.code}")
print(f"  Period range: ({inv.start}, {inv.end})")

# Try end-of-month date (D3 absorbs remainder)
inv2 = dekadal.inverse("2025-02-28")
print(f"\nDate 2025-02-28 -> member {inv2.code}")
print(f"  Period range: ({inv2.start}, {inv2.end})")

## 2. Pentadal Generator (5-day periods)

A **pentad** divides each month into six 5-day periods:
- P1: days 1-5, P2: days 6-10, ..., P5: days 21-25
- P6: days 26-end of month (absorbs remainder: 1, 2, 3, 4, or 6 days)

This gives 72 periods per year. Used by CHIRPS, CDT, FAO climate products.

In [ ]:
pentadal = DailyPeriodProvider(period_days=5, scheme="monthly")

print(f"Provider type:  {pentadal.provider_type}")
print(f"Config:         {pentadal.config_as_dict()}")

# Generate all pentads for January 2025
result = pentadal.generate("2025-01-01", "2025-01-31", limit=10)
print(f"\nJanuary 2025 pentads ({result.number_matched} total):")
for m in result.members:
    print(f"  {m.code}  {m.start} -> {m.end}")

In [ ]:
# Inverse: which pentad contains Jan 27?
inv = pentadal.inverse("2025-01-27")
print(f"Date 2025-01-27 -> member {inv.code}")
print(f"  Period range: ({inv.start}, {inv.end})")

## 3. Admin Boundary Hierarchy (LeveledTreeProvider)

Hierarchical nominal dimensions model parent-child relationships:
**Continent -> Country -> Region**

The `LeveledTreeProvider` supports:
- Root member listing
- Children navigation (`children(parent_code)`)
- Ancestor traversal (`ancestors(code)`)
- Multilingual labels
- Text search across labels

In [ ]:
from ogc_dimensions.providers import LeveledTreeProvider
from dynastore.extensions.dimensions.use_cases import ADMIN_NODES

admin = LeveledTreeProvider(nodes=ADMIN_NODES)

print(f"Provider type:  {admin.provider_type}")
print(f"Hierarchical:   {admin.hierarchical}")
print(f"Capabilities:   {admin.capabilities}")
print(f"Search protos:  {admin.search_protocols}")

In [ ]:
# List root members (continents)
continents = admin.generate("", "", limit=10)
print(f"Root members ({continents.number_matched} total):")
for m in continents.members:
    print(f"  {m.code}  label={m.extra.get('label')}  has_children={m.has_children}")

In [ ]:
# Children of Africa, sorted by French label
afr_children = admin.children("AFR", sort_by="label", language="fr")
print(f"Children of AFR ({afr_children.number_matched} total, sorted by French label):")
for m in afr_children.members:
    fr_label = m.extra.get("labels", {}).get("fr", m.extra.get("label"))
    print(f"  {m.code}  fr={fr_label}")

In [ ]:
# Ancestors of Ethiopia (Tigray -> Ethiopia -> Africa)
ancestors = admin.ancestors("ETH")
print(f"Ancestors of ETH: {[n['code'] for n in ancestors]}")

# Ancestors of a region
ancestors_region = admin.ancestors("ETH-TIG")
print(f"Ancestors of ETH-TIG: {[n['code'] for n in ancestors_region]}")

In [ ]:
# Search across all labels
search = admin.search(
    protocol=admin.search_protocols[1],  # LIKE
    extent_min="",
    extent_max="",
    like="*Tan*",
    language="en",
)
print(f"Search 'Tan*': {[m.code for m in search.members]}")

## 4. Indicator Hierarchy

The same `LeveledTreeProvider` models statistical indicator catalogs:
**Domain -> Group -> Indicator**

Example: Food Security -> Availability -> Dietary Energy Supply

In [ ]:
from dynastore.extensions.dimensions.use_cases import INDICATOR_NODES

indicators = LeveledTreeProvider(nodes=INDICATOR_NODES)

# Root domains
domains = indicators.generate("", "", limit=10)
print(f"Indicator domains ({domains.number_matched}):")
for m in domains.members:
    print(f"  {m.code}  {m.extra.get('label')}")

# Drill into Food Security
print("\nFood Security groups:")
fs_children = indicators.children("FS")
for m in fs_children.members:
    print(f"  {m.code}  {m.extra.get('label')}")

# Drill into Availability
print("\nAvailability indicators:")
avl_children = indicators.children("FS-AVL")
for m in avl_children.members:
    unit = m.extra.get("unit", "")
    print(f"  {m.code}  {m.extra.get('label')}  [{unit}]")

## 5. Registering Dimensions via API

On a running GeoID instance, dimensions are registered through the Dimensions extension REST API.

### Register a dekadal temporal dimension

```http
PUT /dimensions/temporal-dekadal
Content-Type: application/json

{
  "provider_type": "daily-period",
  "config": {
    "period_days": 10,
    "scheme": "monthly"
  }
}
```

### Register an admin boundary hierarchy

```http
PUT /dimensions/admin-boundaries
Content-Type: application/json

{
  "provider_type": "leveled-tree",
  "config": {
    "source_table": "gaul_admin",
    "code_field": "adm_code",
    "label_field": "adm_name",
    "parent_field": "parent_code",
    "level_field": "adm_level"
  }
}
```

Once registered, the dimension exposes paginated endpoints:
- `GET /dimensions/temporal-dekadal/items?limit=10&offset=0`
- `GET /dimensions/temporal-dekadal/inverse?value=2024-01-15`
- `GET /dimensions/admin-boundaries/children?parent=AFR`
- `GET /dimensions/admin-boundaries/ancestors?code=ETH`

In [ ]:
# Example: register a dekadal dimension via HTTP (uncomment to run against a live instance)

# import httpx
#
# BASE_URL = "https://data.review.fao.org/geospatial/search"
#
# response = httpx.put(
#     f"{BASE_URL}/dimensions/temporal-dekadal",
#     json={
#         "provider_type": "daily-period",
#         "config": {"period_days": 10, "scheme": "monthly"},
#     },
#     headers={"Authorization": "Bearer <your-token>"},
# )
# print(response.status_code, response.json())

## Summary

| Provider | `provider_type` | Hierarchical | Capabilities |
|-----------|------------------|--------------|-------------|
| `DailyPeriodProvider` | `daily-period` | No | generate, extent, inverse, search |
| `LeveledTreeProvider` | `leveled-tree` | Yes | generate, children, ancestors, search |
| `IntegerRangeProvider` | `integer-range` | No | generate, extent, inverse |

All providers produce `ProducedMember` objects with:
- `code` — unique identifier within the dimension
- `start` / `end` — temporal or ordinal bounds
- `extra` — additional properties (labels, units, etc.)
- `has_children` — for hierarchical providers

Next: See **02_asis_dimensions.ipynb** for a real-world STAC collection using dekadal pagination.